<a href="https://colab.research.google.com/github/ericksbr/grouper/blob/main/grouper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<p align="center">
  <img src="https://raw.githubusercontent.com/ericksbr/grouper/main/images/grouper_logo.png" alt="Grouper logo" width="600"/>
</p>

# Grouper: Find Public Comments with Similar Text

Grouper reads a spreadsheet of public comments and **sorts them into groups of similar comments** based on text similarity.

Grouper compares **wording, not meaning**. Two comments that say the same thing in different words may not be grouped. Two comments that share long stretches of identical wording almost always will be.

## What you need
- A spreadsheet of comments: CSV, Excel (.xlsx), or Parquet. One row per comment, with the comment text in one column.
- A Google account, so you can open this notebook in Colab (Google's free, web-based Python environment).
- Time: a few minutes for a file of a few thousand comments. Larger files take longer. Progress bars show how far along a stage is.

## What you will get
Each comment gets a label:
- **Ungrouped:** no other comment is similar enough.
- **Grouped - Identical:** exactly matches another comment once capitalization and punctuation are ignored.
- **Grouped - Highly similar:** its wording overlaps at least 90% with its group's representative (Jaccard similarity score of 0.90 or higher, on a scale from 0 to 1).
- **Grouped - Moderately similar:** its wording overlaps at least 50% with its group's representative (Jaccard score of 0.50 or higher).

Groups that are variations of the same template can be linked into a **family**.

## How this notebook is organized
Work through the parts in order:
1. **Steps 1–3: Get ready.** Install the tools, define the helper functions, upload your file, and check the columns.
2. **Act 1: Analyze.** Grouper does the work of identifying similar comments, one stage at a time.
3. **Intermission: Save.** Download your results, so you have a copy to come back to.
4. **Act 2: Explore.** Look at charts, then browse the actual comments by group.

## Colab basics (read this first)
Colab runs the code on a computer in Google's cloud. You don't install anything on your own computer.

- **Code cells:** the boxes with code in them. Text cells, like this one, are not code.
- **Running a cell:** hover over the top-left corner of a code cell until a **play button** (▶) appears, then click it. You can also click inside a cell and press **Shift + Enter**.
- **Knowing a cell is done:** a check mark appears beside the cell (left side) when it finishes, and the spinning icon stops. Each step says what "done" looks like.
- **Order matters:** run cells from top to bottom. Don't use "Run all" from the Runtime menu, because the upload and column steps need you to act.
- **Collapsing a section:** click the **arrow (>)** next to a section heading to hide its cells. Click it again to show them. Collapse sections once you're finished with them.
- **Uploading a file:** when a cell asks for a file, a **Choose Files** button appears (some browsers label it **Browse**). Click it, find your file, and select it.
- **If the session disconnects:** Colab disconnects after a period of inactivity. If that happens, start again from Step 1. If Colab says it can't allocate a runtime, wait a while and try again.

---

Ready? Start with Step 1 below.

# Step 1 — Get ready (about 1–2 minutes)

This step installs the tools Grouper needs. You only need to do it once per session.

1.  Collapse this section by clicking the down arrow at left (˅).
2.  Click the **play button** (next to 16 cells hidden) on the code cell below. If asked, say "Run anyway".
3.  Wait. If you want, you can expand by clicking the side arrow (>). You'll see lines of text as each tool installs, then the message **✓ Setup complete (Google Colab)**.
4.  **Done when:** you see "Setup complete" and the check mark appears for all code cells in Step 1. Go to Step 2.

In [ ]:
# Install dependencies
import subprocess
import sys

packages = ['pandas', 'numpy', 'scipy', 'scikit-learn', 'openpyxl', 'pyarrow', 'nltk', 'matplotlib', 'tqdm', 'datasketch>=2.0.0,<3', 'psutil']
for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

# Imports
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
from collections import Counter
from difflib import SequenceMatcher
import unicodedata
import re
import time
import gc
import datasketch
from datasketch import MinHash, MinHashLSH
assert tuple(int(p) for p in datasketch.__version__.split('.')[:2]) >= (2, 0), \
    f'datasketch 2.x required, found {datasketch.__version__}'
from ipywidgets import Dropdown, Output, IntText, VBox, HBox, HTML
from IPython.display import display, clear_output
from scipy.sparse import csr_matrix
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Download NLTK data
import nltk
try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab', quiet=True)

# Google Colab file upload
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"✓ Setup complete ({'Google Colab' if IN_COLAB else 'Local Jupyter'})")

## Define helper functions

The cell below defines the functions that read your file and guess which column holds the comments. Click play, wait for it to finish, and move on. You don't need to change anything here.

In [ ]:
# Function: Load CSV, XLSX, or Parquet file
def load_file(filepath):
    """Load CSV/XLSX/Parquet file as DataFrame."""
    ext = Path(filepath).suffix.lower()
    if ext == '.csv':
        return pd.read_csv(filepath, dtype=str, keep_default_na=False)
    elif ext in ['.xlsx', '.xls']:
        return pd.read_excel(filepath, dtype=str, keep_default_na=False)
    elif ext == '.parquet':
        return pd.read_parquet(filepath).astype(str)
    else:
        raise ValueError(f"Unsupported file type: {ext}")

# Function: Detect which column has comments
def detect_comment_column(df):
    """Auto-detect comment column (longest text on average)."""
    return max(df.columns, key=lambda c: df[c].astype(str).str.len().mean())

# Function: Detect which column has a commenter id
def detect_id_column(df):
    """Auto-detect ID column: any column with 'id' as a word in its name (e.g. 'commenter_id', 'Document ID', 'documentId')."""
    def name_tokens(name):
        spaced = re.sub(r'([a-z0-9])([A-Z])', r'\1 \2', str(name))  # split camelCase
        return [t for t in re.split(r'[^a-z0-9]+', spaced.lower()) if t]

    candidates = []
    for col in df.columns:
        tokens = name_tokens(col)
        if 'id' in tokens or 'ids' in tokens or 'tracking' in tokens:
            candidates.append(col)
    if not candidates:
        return None

    # Prefer commenter-style names, then document/tracking, then plain 'id'
    priority = ['commenter', 'comment', 'document', 'tracking', 'id']
    def rank(col):
        tokens = name_tokens(col)
        return min((priority.index(t) for t in tokens if t in priority), default=len(priority))
    return min(candidates, key=rank)

In [ ]:
def on_comment_change(change):
    global comment_col
    comment_col = change['new']
    update_display()

def on_id_change(change):
    global id_col
    id_col = None if change['new'] == '(none)' else change['new']
    update_display()

def update_display():
    with output:
        output.clear_output()
        sample = df[comment_col].fillna("").astype(str).iloc[0]
        print(f"âœ“ Comment column: {comment_col}")
        print(f"âœ“ ID column: {id_col or '(row index)'}")
        print(f"\nSample ({len(sample)} chars):")
        print(f"{sample[:300]}...")

In [ ]:
def mark_stage(name):
    """Record seconds since the previous stage mark under this name."""
    global t_stage
    now = time.perf_counter()
    run_stages[name] = round(now - t_stage, 3)
    t_stage = now

In [ ]:
# Function: Exact match on raw text
def exact_match_raw(texts):
    """Exact match on raw text."""
    df_raw = pd.DataFrame({'text': texts})
    exact_grp_raw = df_raw.groupby('text', sort=False).ngroup() + 1
    is_first_seen_raw = ~df_raw.duplicated('text', keep='first').values
    return exact_grp_raw.values, is_first_seen_raw

In [ ]:
# Function: Normalize text
def normalize_text(text: str) -> str:
    """Normalize text: NFKC, lowercase, NLTK tokenize, alphanumeric filter."""
    text = unicodedata.normalize('NFKC', text)
    text = text.lower()
    tokens = nltk.word_tokenize(text)
    tokens = [t for t in tokens if any(c.isalnum() for c in t)]
    text = ' '.join(tokens)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Function: Exact match on normalized text
def exact_match_normalized(texts, is_first_seen_raw):
    """Exact match on normalized text (efficient: normalize first-seen-raw only)."""
    normalized_texts = np.empty(len(texts), dtype=object)
    raw_to_norm = {}

    # Normalize only first-seen-raw
    for idx in tqdm(np.where(is_first_seen_raw)[0], desc='Normalizing', unit='doc'):
        raw_text = texts[idx]
        norm_text = normalize_text(raw_text)
        normalized_texts[idx] = norm_text
        raw_to_norm[raw_text] = norm_text

    # Propagate to exact raw-text duplicates
    for idx in np.where(~is_first_seen_raw)[0]:
        raw_text = texts[idx]
        normalized_texts[idx] = raw_to_norm.get(raw_text, "")

    # Group on normalized text
    df_norm = pd.DataFrame({'text_normalized': normalized_texts})
    exact_grp_normalized = df_norm.groupby('text_normalized', sort=False).ngroup() + 1
    is_first_seen_normalized = ~df_norm.duplicated('text_normalized', keep='first').values

    return normalized_texts, exact_grp_normalized.values, is_first_seen_normalized

In [ ]:
# Function: Build a matrix of shingles (n-word segments from document)
def build_shingle_matrix(texts, k=5):
    """Build sparse shingle matrix (docs — shingles)."""
    all_shingles_set = set()
    doc_shingles = []

    for text in texts:
        words = text.split()
        if len(words) >= k:
            shingles = {' '.join(words[i:i+k]) for i in range(len(words) - k + 1)}
        else:
            shingles = {text}
        doc_shingles.append(shingles)
        all_shingles_set.update(shingles)

    shingle_list = sorted(list(all_shingles_set))
    shingle_to_idx = {s: i for i, s in enumerate(shingle_list)}

    rows, cols, data = [], [], []
    for doc_idx, shingles in enumerate(doc_shingles):
        for shingle in shingles:
            rows.append(doc_idx)
            cols.append(shingle_to_idx[shingle])
            data.append(1)

    X = csr_matrix((data, (rows, cols)), shape=(len(doc_shingles), len(shingle_list)), dtype=np.uint8)
    return X

# Function: Compute MinHash signatures (datasketch, affine32 scheme)
def _identity(x):
    """Shingle ids are already integers below 2**32, so they serve as hash values."""
    return x

def minhash_signatures(X, num_perm=256, seed=42):
    """MinHash signatures via datasketch. Each doc's shingle ids are fed in as hash inputs."""
    n_docs = X.shape[0]
    assert X.shape[1] < 2**32, 'Too many shingles for 32-bit hash inputs'
    assert np.all(np.diff(X.indptr) > 0), 'Every doc needs at least one shingle'

    sig = np.empty((n_docs, num_perm), dtype=np.uint32)
    docs = (X.indices[X.indptr[d]:X.indptr[d + 1]].tolist() for d in range(n_docs))
    mhs = MinHash.generator(docs, num_perm=num_perm, seed=seed, scheme='affine32', hashfunc=_identity)
    for d, mh in enumerate(tqdm(mhs, total=n_docs, desc='MinHash', unit='doc')):
        sig[d] = mh.hashvalues
    return sig


In [ ]:
def choose_lsh_params(num_perm, min_threshold, target_recall):
    """Largest rows-per-band r (fewest junk candidates) whose S-curve still gives
    P(candidate | J = min_threshold) >= target_recall."""
    best = None
    for r in range(1, num_perm + 1):
        b = num_perm // r
        if 1 - (1 - min_threshold**r) ** b >= target_recall:
            best = (b, r)
    if best is None:
        raise ValueError('No (b, r) reaches the target recall; raise NUM_PERM or lower TARGET_RECALL')
    return best

In [ ]:
# Greedy claiming: each comment joins a representative or starts a group, in the order given.
def band_keys(sig, b, r):
    """One hash per comment per LSH band. Comments with the same key share a bucket in that band."""
    return [pd.util.hash_pandas_object(pd.DataFrame(sig[:, band * r:(band + 1) * r]), index=False).values
            for band in range(b)]

def assign_greedy(sig, X, b, r, t, K=5, order=None):
    """Greedy star clustering.
    Pass 1: a comment that is still unassigned starts a group (becomes its representative) and claims the
    unassigned comments it matches (exact J >= t) among its LSH candidates. A comment does not start a group
    if every one of its buckets already has K representatives (K caps the representatives per bucket).
    Pass 2: every comment still unassigned starts its own group the same way, so nothing is left out.
    A comment joins the first representative that matches it (no argmax). Representatives anchor to themselves.
    Returns (anchor, anchor_j): the group representative of each comment and its J to that representative."""
    n = sig.shape[0]
    if order is None:
        order = np.arange(n)
    Xf = X.astype(np.float32)  # float, so the sparse products cannot overflow
    sizes = np.diff(X.indptr).astype(np.float32)
    lsh = MinHashLSH(num_perm=b * r, params=(b, r))
    template = MinHash(num_perm=b * r, seed=42, scheme='affine32')

    def mh_of(d):
        mh = template.copy()
        mh.hashvalues = sig[d, :b * r].copy()
        return mh

    for d in tqdm(range(n), desc='Indexing', unit='doc'):
        lsh.insert(d, mh_of(d), check_duplication=False)

    keys = band_keys(sig, b, r)
    reps_in_bucket = [Counter() for _ in range(b)]
    anchor = np.full(n, -1, dtype=np.int64)
    anchor_j = np.zeros(n, dtype=np.float32)

    def bucket_full(d):
        return all(reps_in_bucket[band][keys[band][d]] >= K for band in range(b))

    def start_group(d):
        anchor[d] = d
        anchor_j[d] = 1.0
        for band in range(b):
            reps_in_bucket[band][keys[band][d]] += 1
        cands = np.array([c for c in lsh.query(mh_of(d)) if anchor[c] == -1], dtype=np.int64)
        if cands.size:
            inter = (Xf[cands] @ Xf[d].T).toarray().ravel()
            j = inter / (sizes[cands] + sizes[d] - inter)
            take = j >= t - 1e-9
            anchor[cands[take]] = d
            anchor_j[cands[take]] = j[take]

    for d in tqdm(order, desc='Pass 1: capped reps', unit='doc'):
        if anchor[d] == -1 and not bucket_full(d):
            start_group(d)
    for d in tqdm(order, desc='Pass 2: remaining', unit='doc'):
        if anchor[d] == -1:
            start_group(d)
    del lsh
    return anchor, anchor_j

In [ ]:
def group_ids_from_anchor(anchor):
    """Number groups 1..G, largest first; each doc takes its anchor's number."""
    _, inverse, sizes = np.unique(anchor, return_inverse=True, return_counts=True)
    rank_order = np.argsort(-sizes, kind='stable')
    rank = np.empty(len(sizes), dtype=np.int32)
    rank[rank_order] = np.arange(1, len(sizes) + 1, dtype=np.int32)
    return rank[inverse]

In [ ]:
def assign_group_label(is_identical, max_j_in_group, group_size, HIGH_J=0.90, GROUP_J=0.50):
    """Assign group label based on similarity to best match in group."""
    if group_size == 1:
        return 'Ungrouped'
    elif is_identical:
        return 'Grouped - Identical'
    elif max_j_in_group >= HIGH_J:
        return 'Grouped - Highly similar'
    elif max_j_in_group >= GROUP_J:
        return 'Grouped - Moderately similar'
    else:
        # Should not happen if grouping was done at GROUP_J threshold
        return 'Ungrouped'

In [ ]:
def get_jaccard_similarity(norm_text1, norm_text2, k=5):
    """Jaccard similarity on k-word shingles (assumes already-normalized text).

    Args:
        norm_text1, norm_text2: already normalized text (whitespace-separated words)
        k: shingle size in words

    Returns:
        Jaccard similarity in [0, 1]
    """
    words1 = norm_text1.split()
    words2 = norm_text2.split()

    # For documents with < k words, use whole text as one shingle (consistent with build_shingle_matrix)
    if len(words1) < k:
        shingles1 = {' '.join(words1)} if words1 else {''}
    else:
        shingles1 = {' '.join(words1[i:i+k]) for i in range(len(words1) - k + 1)}

    if len(words2) < k:
        shingles2 = {' '.join(words2)} if words2 else {''}
    else:
        shingles2 = {' '.join(words2[i:i+k]) for i in range(len(words2) - k + 1)}

    intersection = len(shingles1 & shingles2)
    union = len(shingles1 | shingles2)
    return intersection / union if union > 0 else 0.0


In [ ]:
BUCKET_LABELS = {
    'near_exact': 'Near-Exact (J > 0.95)',
    'high_var': 'High Similarity (0.80 - 0.95)',
    'med_var': 'Medium Similarity (0.60 - 0.80)',
    'low_var': 'Low Similarity (J < 0.60)',
    'all': 'All Documents'
}

def get_bucket_label(bucket_name):
    """Get descriptive label for bucket."""
    return BUCKET_LABELS.get(bucket_name, bucket_name)

def compare_docs_side_by_side(doc1_text, doc2_text):
    """Word-level diff and Jaccard similarity."""
    words1 = str(doc1_text).split()
    words2 = str(doc2_text).split()

    matcher = SequenceMatcher(None, words1, words2)
    word_sim = matcher.ratio()
    jaccard_sim = get_jaccard_similarity(normalize_text(str(doc1_text)), normalize_text(str(doc2_text)), k=5)

    left_html = right_html = ''
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == 'equal':
            left_html += ' '.join(words1[i1:i2]) + ' '
            right_html += ' '.join(words2[j1:j2]) + ' '
        elif tag == 'delete':
            left_html += f'<span style="background: #ffcccc; text-decoration: line-through;">{" ".join(words1[i1:i2])}</span> '
        elif tag == 'insert':
            right_html += f'<span style="background: #ccffcc;">{" ".join(words2[j1:j2])}</span> '
        elif tag == 'replace':
            left_html += f'<span style="background: #ffcccc;">{" ".join(words1[i1:i2])}</span> '
            right_html += f'<span style="background: #ccffcc;">{" ".join(words2[j1:j2])}</span> '

    return jaccard_sim, word_sim, left_html, right_html

def build_explorer(results, comment_col, id_col):
    """Simplified explorer: groups sorted by J_to_rep."""
    # Filter to multi-member groups
    multi_groups = results.groupby('group_id').filter(lambda x: len(x) > 1)

    if len(multi_groups) == 0:
        print("⊘ No groups with 2+ members to explore")
        return

    # Build group list (sorted by size, descending)
    group_ids = multi_groups['group_id'].unique()
    group_options = {
        f"Group {int(gid)} · {(multi_groups['group_id'] == gid).sum()} copies": gid
        for gid in sorted(group_ids, key=lambda g: (multi_groups['group_id'] == g).sum(), reverse=True)
    }

    group_dropdown = Dropdown(
        options=group_options,
        value=list(group_options.values())[0],
        description='Group:',
        layout={'width': '600px'}
    )

    doc_idx_1 = IntText(value=0, description='Doc 1:', min=0, max=1000, layout={'width': '250px'})
    doc_idx_2 = IntText(value=1, description='Doc 2:', min=0, max=1000, layout={'width': '250px'})

    output_area = Output()

    def update_group(change):
        """Update doc list when group changes (sorted by J desc)."""
        group_id = change['new']
        group_rows = multi_groups[multi_groups['group_id'] == group_id].sort_values('J_to_rep', ascending=False)

        max_idx = len(group_rows) - 1
        doc_idx_1.max = max_idx
        doc_idx_2.max = max_idx
        doc_idx_1.value = 0
        doc_idx_2.value = min(1, max_idx)

        explore(group_id, 0, 1, group_rows)

    def explore(group_id, d1_idx, d2_idx, group_rows):
        """Display two docs side-by-side."""
        with output_area:
            clear_output(wait=True)

            group_rows_sorted = group_rows.sort_values('J_to_rep', ascending=False)

            if len(group_rows_sorted) < 2:
                display(HTML("<p style='color: orange;'>⊘ Not enough documents to compare</p>"))
                return

            d1_idx = min(max(d1_idx, 0), len(group_rows_sorted) - 1)
            d2_idx = min(max(d2_idx, 0), len(group_rows_sorted) - 1)

            if d1_idx == d2_idx:
                d2_idx = (d1_idx + 1) % len(group_rows_sorted)

            row1 = group_rows_sorted.iloc[d1_idx]
            row2 = group_rows_sorted.iloc[d2_idx]

            jaccard_sim, word_sim, left_html, right_html = compare_docs_side_by_side(
                row1[comment_col],
                row2[comment_col]
            )

            id1 = row1[id_col] if id_col else f"Doc {d1_idx}"
            id2 = row2[id_col] if id_col else f"Doc {d2_idx}"

            html = f"""
            <div style='font-family: sans-serif; font-size: 13px;'>
            <h4>Group {int(group_id)} (sorted by J to representative)</h4>

            <p style='margin: 8px 0; color: #666;'>
                <b>Jaccard:</b> <span style='color: #2E86AB; font-weight: bold;'>{jaccard_sim:.1%}</span>·
                <b>Word-level:</b> <span style='color: #2E86AB; font-weight: bold;'>{word_sim:.1%}</span>
            </p>

            <p style='margin: 4px 0; color: #666; font-size: 12px;'>
                Doc {d1_idx + 1} of {len(group_rows_sorted)} vs Doc {d2_idx + 1} of {len(group_rows_sorted)}
            </p>

            <hr style='margin: 8px 0; border: none; border-top: 1px solid #ddd;'>

            <table style='width: 100%; border-collapse: collapse; font-size: 12px;'>
            <tr style='background: #f0f0f0;'>
                <th style='text-align: left; padding: 6px; border: 1px solid #ccc;'>{id1}</th>
                <th style='text-align: left; padding: 6px; border: 1px solid #ccc;'>{id2}</th>
            </tr>
            <tr>
                <td style='padding: 8px; border: 1px solid #ddd; background: #fafafa; word-wrap: break-word; line-height: 1.5; max-height: 300px; overflow-y: auto;'>{left_html}</td>
                <td style='padding: 8px; border: 1px solid #ddd; background: #fafafa; word-wrap: break-word; line-height: 1.5; max-height: 300px; overflow-y: auto;'>{right_html}</td>
            </tr>
            </table>
            </div>
            """
            display(HTML(html))

    def on_group_change(change):
        update_group(change)

    def on_doc_idx_change(change):
        group_id = group_dropdown.value
        group_rows = multi_groups[multi_groups['group_id'] == group_id]
        explore(group_id, doc_idx_1.value, doc_idx_2.value, group_rows)

    group_dropdown.observe(on_group_change, names='value')
    doc_idx_1.observe(on_doc_idx_change, names='value')
    doc_idx_2.observe(on_doc_idx_change, names='value')

    display(VBox([
        HTML("<h3>Interactive Explorer: Groups & Comparisons</h3>"),
        HTML(f"<p style='font-size: 0.9em; color: #666;'>{len(group_ids)} groups with 2+ documents</p>"),
        group_dropdown,
        HBox([doc_idx_1, doc_idx_2]),
    ]))
    display(output_area)

    # Initialize first group
    group_id = group_dropdown.value
    group_rows = multi_groups[multi_groups['group_id'] == group_id].sort_values('J_to_rep', ascending=False)
    explore(group_id, 0, 1, group_rows)

In [ ]:
# family explorer
def family_members(family_rows, fid, family_center_doc, normalized_texts):
    rows = family_rows[family_rows['family_id'] == fid]
    anchor = family_center_doc[rows.index[0]]
    reps = rows[rows['is_group_rep']]
    scored = sorted(
        ((get_jaccard_similarity(normalized_texts[doc], normalized_texts[anchor]), doc, row)
         for doc, row in reps.iterrows() if doc != anchor),
        key=lambda t: -t[0]
    )
    options = {
        f"Group {int(row['group_id'])} · {int(row['group_size'])} copies · J to anchor {j:.0%}": doc
        for j, doc, row in scored
    }
    return anchor, options

def family_pair_html(results, comment_col, id_col, normalized_texts, a, b):
    _, word_sim, left_html, right_html = compare_docs_side_by_side(
        results.at[a, comment_col], results.at[b, comment_col]
    )
    jac = get_jaccard_similarity(normalized_texts[a], normalized_texts[b])
    label_a = results.at[a, id_col] if id_col else f"Row {a + 1}"
    label_b = results.at[b, id_col] if id_col else f"Row {b + 1}"
    return f"""
    <div style='font-family: sans-serif; font-size: 13px;'>
    <p style='margin: 8px 0; color: #666;'>
        <b>Jaccard (normalized):</b> <span style='color: #2E86AB; font-weight: bold;'>{jac:.1%}</span> ·
        <b>Word-level:</b> <span style='color: #2E86AB; font-weight: bold;'>{word_sim:.1%}</span>
    </p>
    <table style='width: 100%; border-collapse: collapse; font-size: 12px;'>
    <tr style='background: #f0f0f0;'>
        <th style='text-align: left; padding: 6px; border: 1px solid #ccc;'>Anchor · Group {int(results.at[a, 'group_id'])} · {label_a}</th>
        <th style='text-align: left; padding: 6px; border: 1px solid #ccc;'>Group {int(results.at[b, 'group_id'])} · {label_b}</th>
    </tr>
    <tr>
        <td style='padding: 8px; border: 1px solid #ddd; background: #fafafa; word-wrap: break-word; line-height: 1.5;'>{left_html}</td>
        <td style='padding: 8px; border: 1px solid #ddd; background: #fafafa; word-wrap: break-word; line-height: 1.5;'>{right_html}</td>
    </tr>
    </table>
    </div>
    """

def build_family_explorer(results, comment_col, id_col, family_center_doc, normalized_texts):
    family_rows = results[results['family_id'].notna()]
    fam_info = family_rows.groupby('family_id').agg(docs=('group_id', 'size'), groups=('group_id', 'nunique'))
    fam_info = fam_info[fam_info['groups'] > 1].sort_values('docs', ascending=False)

    if len(fam_info) == 0:
        print("⊘ No families spanning 2+ groups to explore")
        return

    fam_dropdown = Dropdown(
        options={f"Family {fid} · {r.groups} groups · {r.docs} docs": fid for fid, r in fam_info.iterrows()},
        description='Family:',
        layout={'width': '600px'}
    )
    anchor_html = HTML()
    compare_dropdown = Dropdown(description='Compare with:', layout={'width': '600px'})
    fam_output = Output()
    state = {'busy': False, 'anchor': None}

    def refresh(change=None):
        if state['busy'] or state['anchor'] is None or compare_dropdown.value is None:
            return
        with fam_output:
            clear_output(wait=True)
            display(HTML(family_pair_html(results, comment_col, id_col, normalized_texts,
                                          state['anchor'], compare_dropdown.value)))

    def on_family(change):
        state['busy'] = True
        anchor, options = family_members(family_rows, change['new'], family_center_doc, normalized_texts)
        state['anchor'] = anchor
        anchor_row = results.loc[anchor]
        anchor_html.value = (
            f"<p style='font-size: 13px;'><b>Pinned anchor:</b> Group {int(anchor_row['group_id'])} · "
            f"{int(anchor_row['group_size'])} copies</p>"
        )
        compare_dropdown.options = options
        compare_dropdown.value = next(iter(options.values()), None)
        state['busy'] = False
        refresh()

    fam_dropdown.observe(on_family, names='value')
    compare_dropdown.observe(refresh, names='value')

    display(VBox([
        HTML("<h3>Family Explorer</h3>"),
        HTML("<p style='font-size: 0.9em; color: #666;'>The family anchor is pinned on the left. "
             "Pick any other group to compare it against; groups are listed by similarity to the anchor.</p>"),
        fam_dropdown,
        anchor_html,
        compare_dropdown,
    ]))
    display(fam_output)
    on_family({'new': fam_dropdown.value})


# Step 2 — Upload your comment file

1. Click the **play button** on the code cell below. A **Choose Files / Browse** button appears under the cell.
2. Click **Choose Files** (or **Browse**), find your comment file on your computer, and select it.
3. Wait for the upload to finish.
4. **Done when:** you see a line like "Loaded 12,016 rows; 34 columns." (The number will match your file.)

In [ ]:
# Run widget to find and load your data
if IN_COLAB:
    print("Click 'Choose Files' to upload your CSV/XLSX/Parquet file:")
    uploaded = files.upload()
    filepath = list(uploaded.keys())[0]
else:
    filepath = input("Enter path to your file (CSV/XLSX/Parquet): ")

print(f"File: {filepath}")

t_load = time.perf_counter()
df = load_file(filepath)
load_seconds = time.perf_counter() - t_load
print(f"Loaded {len(df):,} rows; {len(df.columns)} columns")

## Step 3 — Check the columns

Grouper guesses which column holds the comment text and which holds a commenter ID. Click **play** on the cell below. It prints its guesses:

- **Comment column:** the column with the comment text.
- **ID column:** a column such as Document ID or CommentId. If none is found, it says "(row index)".

**Done when:** both guesses are printed. If they're right, skip Step 3b and go on to Act 1. If they're wrong, do Step 3b.

In [ ]:
comment_col = detect_comment_column(df)
id_col = detect_id_column(df)

print(f"Comment column: {comment_col}")
print(f"ID column: {id_col or '(row index)'}")

## Step 3b — Fix the column guesses (optional)

Only needed if Step 3 guessed wrong. Click **play** on the cell below. Two dropdown menus appear, one for the comment column and one for the ID column. Pick the correct column from each list, or choose **(none)** if your file has no ID column. Under the menus, a sample comment is shown. If it reads like a real comment, the comment column is right.

The cell after the dropdowns is optional. It saves a faster Parquet copy of your file for next time. You can skip it.

In [ ]:
comment_dropdown = Dropdown(
    options=df.columns.tolist(),
    value=comment_col,
    description='Comment Column:',
    layout={'width': '500px'}
)

id_dropdown = Dropdown(
    options=['(none)'] + df.columns.tolist(),
    value=id_col if id_col else '(none)',
    description='ID Column:',
    layout={'width': '500px'}
)

output = Output()

comment_dropdown.observe(on_comment_change, names='value')
id_dropdown.observe(on_id_change, names='value')

display(VBox([
    HTML("<h3>Confirm Column Selection</h3>"),
    HTML("<p>Update if auto-detection is wrong:</p>"),
    comment_dropdown,
    id_dropdown,
    output
]))

update_display()

In [ ]:
# Optional - convert to parquet for faster load next time
parquet_path = Path(filepath).with_suffix('.parquet')
if not parquet_path.exists():
    df.to_parquet(parquet_path, index=False)

# Act 1 — Analyze

Here Grouper does the work of identifying similar comments. The stages below run in order. Each stage prints progress, and long stages show a progress bar. A stage is done when its last line is printed and the check mark appears.

1. Prepare the comments
2. Find exact copies
3. Find copies that differ only in capitalization or punctuation
4. Build fingerprints
5. Group similar comments
6. Record each comment's group and score
7. Link groups into families
8. Summary

The easiest way to run all cells is to collapse Act 1 by clicking on the down arrow (˅). Then, click play next to "22 cells hidden". Once running, you can expand the Act 1 section to examine progress.

## Stage 1 — Prepare the comments

Grouper reads the comment text from the column you confirmed and counts the comments. **Done when:** it prints the number of comments, plus their shortest, longest, and average length.

In [ ]:
texts = df[comment_col].fillna("").astype(str).values
n_docs = len(texts)

print(f"{n_docs:,} documents")
print(f"  - Min length: {min((len(t) for t in texts), default=0)} chars")
print(f"  - Max length: {max((len(t) for t in texts), default=0):,} chars")
print(f"  - Mean length: {np.mean([len(t) for t in texts]):.0f} chars")

from datetime import datetime

run_started_at = datetime.now()
t_start = time.perf_counter()
t_stage = t_start
run_stages = {}

## Stage 2 — Find exact copies

Counts comments that are word-for-word identical to another comment. Many form letters are copied exactly. **Done when:** it prints "Identical (raw text)" with a count and a percentage.

In [ ]:
exact_grp_raw, is_first_seen_raw = exact_match_raw(texts)
exact_raw_count = (~is_first_seen_raw).sum()

print(f"Identical (raw text): {exact_raw_count:,} ({100*exact_raw_count/n_docs:.1f}%)")
print(f"  Distinct texts: {exact_grp_raw.max():,}")

## Stage 3 — Find copies that differ only in capitalization or punctuation

Comments like "I oppose this!" and "i OPPOSE this" differ only in capitalization and punctuation. This stage treats them as the same. **Done when:** it prints "Identical (normalized text)" with a count and a percentage.

In [ ]:
normalized_texts, exact_grp_normalized, is_first_seen_normalized = exact_match_normalized(texts, is_first_seen_raw)
exact_norm_count = (~is_first_seen_normalized).sum()

print(f"Identical (normalized text): {exact_norm_count:,} ({100*exact_norm_count/n_docs:.1f}%)")
print(f"Distinct normalized texts: {exact_grp_normalized.max():,}")
mark_stage('exact_dedupe')


## Stage 4 — Build fingerprints

Each comment is split into overlapping 5-word chunks, called **shingles**. Grouper turns each comment's chunks into a short **fingerprint**, which lets it compare long comments quickly. Comments that share many chunks get similar fingerprints. You don't need the details to use the results.

**Done when:** it prints the size of the shingle matrix and the signatures.

In [ ]:
fs_pos = np.where(is_first_seen_normalized)[0]
fs_texts = normalized_texts[fs_pos]
n_fs = len(fs_pos)

print(f"First-seen (normalized): {n_fs:,}")

X = build_shingle_matrix(fs_texts)
print(f"Shingle matrix: {X.shape[0]} docs; {X.shape[1]:,} shingles")

sig = minhash_signatures(X, num_perm=256)
print(f"\nSignatures: {sig.shape}")
mark_stage('minhash')


## Stage 5 — Group similar comments

Grouper uses the fingerprints to find comments that are likely to be similar, then checks each candidate by counting shared 5-word chunks.

The score it uses is **Jaccard similarity**. It runs from 0 to 1: 0 means no chunks in common, and 1 means identical wording. Grouper only groups two comments when their score is **0.50 or higher**.

The cells below set up the search and run it. **Done when:** the grouping cell prints the number of groups.

In [ ]:
b, r = choose_lsh_params(256, 0.50, 0.995)
print(f"LSH: {b} bands x {r} rows per band")

### How the grouping works (optional reading)

- Comments are visited **most-copied first**, so the most common letter becomes a group's representative.
- A comment that hasn't joined a group yet becomes a representative. It claims every unassigned comment it matches at 0.50 or higher.
- A comment joins the first representative that matches it.
- To keep the work manageable, each fingerprint bucket allows only five representatives in the first pass. A second pass then lets every leftover comment start its own group, so no comment is dropped.
- Representatives always belong to their own group, so groups don't chain together through a middle comment.

"LSH" (locality-sensitive hashing) is the bucket technique that makes finding candidates fast.

In [ ]:
# Run greedy claiming on first-seen comments (exact duplicates are mapped back below)
text_count = Counter(normalized_texts)
copies = np.array([text_count[t] for t in fs_texts])
order = np.lexsort((np.arange(n_fs), -copies))  # most-copied first, then position

K_REPS = 5  # max representatives per LSH bucket in the first pass (per band)
anchor_fs, anchor_j_fs = assign_greedy(sig, X, b, r, 0.50, K=K_REPS, order=order)
mark_stage('lsh_verify')

del X, sig
gc.collect()
print(f"✓ Greedy groups on first-seen comments (incl. singletons): {(anchor_fs == np.arange(n_fs)).sum():,}")

In [ ]:
# Map first-seen results back to all comments. Exact duplicates join their twin's group.
anchor_all = np.full(n_docs, -1, dtype=np.int64)
anchor_all[fs_pos] = fs_pos[anchor_fs]
anchor_j_all = np.zeros(n_docs, dtype=np.float32)
anchor_j_all[fs_pos] = anchor_j_fs

first_idx_of = {normalized_texts[i]: i for i in fs_pos}
twin_src = np.array([first_idx_of[t] for t in normalized_texts])
anchor = anchor_all[twin_src]
anchor_j = anchor_j_all[twin_src]
group_ids = group_ids_from_anchor(anchor)

## Stage 6 — Record each comment's group and score

For every comment, Grouper records:

- **group_id** and **group_size:** which group the comment belongs to and how big that group is.
- **J_to_rep:** its similarity to its group's representative.
- **max_j_in_group:** the group's best score. A representative takes the highest score among its members, and identical comments score 1.0.
- **group_label:** the label described in the About section.

**Done when:** it prints "Set representatives for N groups."

In [ ]:
group_rep_id = anchor.astype(np.int32)
is_group_rep = anchor == np.arange(n_docs)
J_to_rep = anchor_j.astype(np.float32)
is_identical = np.array([text_count[t] > 1 for t in normalized_texts])

nonrep = ~is_group_rep
best_member_j = np.zeros(group_ids.max() + 1, dtype=np.float32)
np.maximum.at(best_member_j, group_ids[nonrep], J_to_rep[nonrep])
max_j_in_group = np.where(is_group_rep, best_member_j[group_ids], J_to_rep).astype(np.float32)
max_j_in_group[is_identical] = 1.0
best_match_id = group_rep_id.copy()
mark_stage('identical_flags')
print(f"✓ Set representatives for {len(np.unique(anchor)):,} groups")

## Stage 7 — Link related groups into families

A **family** is a set of groups that are variations of the same template, such as the same letter with some words changed.

Each group is judged by its **medoid**: the comment most similar to the rest of its group. Grouper compares every pair of medoids. Groups whose medoids share at least 40% of their chunks (Jaccard score of 0.40 or higher) are linked into one family. Groups that aren't linked to any other group are not in a family. Singleton comments are left out.

**Done when:** it prints the number of families.

In [ ]:
# Medoid of each group: the member with the highest average Jaccard to the other members of its group.
# The family pass compares medoids, not the greedy representatives, so the comparison point is central
# to its group. Grouping itself still uses the greedy representatives.
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

X_all = build_shingle_matrix(normalized_texts)
sizes_all = np.diff(X_all.indptr).astype(np.float32)

medoid = np.full(group_ids.max() + 1, -1, dtype=np.int64)
by_group = np.argsort(group_ids, kind='stable')
splits = np.flatnonzero(np.diff(group_ids[by_group])) + 1
for members in tqdm(np.split(by_group, splits), desc='Medoids', unit='group'):
    gid = group_ids[members[0]]
    if len(members) == 1:
        medoid[gid] = members[0]
        continue
    Xg = X_all[members].astype(np.float32)
    sg = sizes_all[members]
    mean_j = np.zeros(len(members), dtype=np.float32)
    for start in range(0, len(members), 1000):
        stop = min(start + 1000, len(members))
        inter = (Xg[start:stop] @ Xg.T).toarray()
        jac = inter / (sg[start:stop, None] + sg[None, :] - inter)
        jac[np.arange(stop - start), np.arange(start, stop)] = 0  # leave out self
        mean_j[start:stop] = jac.sum(axis=1) / (len(members) - 1)
    medoid[gid] = members[np.argmax(mean_j)]

In [ ]:
FAMILY_J = 0.40  # medoids at J >= 0.40 are linked into the same family

group_sizes = np.bincount(group_ids)
fam_groups = np.where(group_sizes > 1)[0]          # groups with 2+ members
fam_docs = medoid[fam_groups]                       # one comparison point per group
X_f = X_all[fam_docs].astype(np.float32)
sz_f = sizes_all[fam_docs]

# Compare every pair of medoids (exact Jaccard), in blocks to keep memory small
R = len(fam_docs)
edge_i, edge_j = [], []
for start in range(0, R, 1000):
    stop = min(start + 1000, R)
    inter = (X_f[start:stop] @ X_f.T).toarray()
    jac = inter / (sz_f[start:stop, None] + sz_f[None, :] - inter)
    ii, jj = np.nonzero(jac >= FAMILY_J - 1e-9)
    ii = ii + start
    keep = ii < jj
    edge_i.append(ii[keep])
    edge_j.append(jj[keep])
edge_i = np.concatenate(edge_i)
edge_j = np.concatenate(edge_j)

# Families are connected groups: two groups are in the same family if a chain of medoid matches links them
graph = coo_matrix((np.ones(len(edge_i)), (edge_i, edge_j)), shape=(R, R))
n_comp, comp = connected_components(graph, directed=False)

comp_size = np.bincount(comp, minlength=n_comp)
linked_comps = np.where(comp_size > 1)[0]
linked_comps = linked_comps[np.argsort(-comp_size[linked_comps], kind='stable')]  # largest family first
fam_number = np.zeros(n_comp, dtype=np.int32)
fam_number[linked_comps] = np.arange(1, len(linked_comps) + 1)

group_family = np.zeros(len(group_sizes), dtype=np.int32)
group_family[fam_groups] = fam_number[comp]
family_id = group_family[group_ids]                 # 0 = not in a family

# Family anchor = medoid of the family's largest group (used by the family explorer)
anchor_of_family = np.full(len(linked_comps) + 1, -1, dtype=np.int64)
for k, c in enumerate(linked_comps, start=1):
    members_g = fam_groups[comp == c]
    anchor_of_family[k] = medoid[members_g[np.argmax(group_sizes[members_g])]]
family_anchor_row = anchor_of_family[family_id]

in_family = family_id > 0
family_id = pd.Series(np.where(in_family, family_id, np.nan)).astype('Int32').values
family_anchor_row = pd.Series(np.where(in_family, family_anchor_row, np.nan)).astype('Int64').values
family_size = pd.Series(family_id).map(pd.Series(family_id).value_counts()).astype('Int32').values

print(f"✓ Family pass: {R:,} group medoids compared (all pairs) → {len(linked_comps):,} families spanning 2+ groups")
mark_stage('family_pass')

## Stage 8 — Summary

Prints the number of comments under each label, plus counts of groups and families. **Done when:** you see "Total rows" with the number of comments. Act 1 is complete.

In [ ]:
total_seconds = time.perf_counter() - t_start

# Build results table with all new columns
results = df.copy()
results['group_id'] = group_ids
results['group_size'] = results['group_id'].map(pd.Series(group_ids).value_counts())
results['J_to_rep'] = J_to_rep
results['max_j_in_group'] = max_j_in_group
results['best_match_id'] = best_match_id
results['is_identical'] = is_identical
results['is_group_rep'] = is_group_rep
results['group_rep_id'] = group_rep_id
results['family_id'] = family_id
results['family_size'] = family_size
results['family_anchor_row'] = family_anchor_row

# Set label order for consistent display
label_order = ['Ungrouped', 'Grouped - Identical', 'Grouped - Highly similar', 'Grouped - Moderately similar']
results['group_label'] = pd.Categorical(
    [assign_group_label(i, m, s) for i, m, s in zip(is_identical, max_j_in_group, results['group_size'])],
    categories=label_order, ordered=True)

print(f"Processing Time: {total_seconds:.1f}s\n")
print("Group labels:")
print(results['group_label'].value_counts(sort=False))
print(f"\nTotal rows: {len(results):,}")

In [ ]:
print("\nFamily statistics:")
fam_rows = results[results['family_id'].notna()]
fam_summary = fam_rows.groupby('family_id').agg(docs=('group_id', 'size'), groups=('group_id', 'nunique'))
multi_fam = fam_summary[fam_summary['groups'] > 1]
print(f"  Families: {len(fam_summary):,} ({len(multi_fam):,} spanning 2+ groups)")

if len(multi_fam) > 0:
    print(f"  Documents in multi-group families: {multi_fam['docs'].sum():,}")
    print(f"  Groups per family: min={multi_fam['groups'].min()}, max={multi_fam['groups'].max()}, mean={multi_fam['groups'].mean():.1f}")
    print("\n  Largest families:")
    print(multi_fam.sort_values('docs', ascending=False).head(10).rename(
        columns={'docs': 'documents', 'groups': 'groups'}).to_string())


In [ ]:
print("\nGroup statistics:")
grouped_docs = results[results['group_size'] > 1]
n_groups = grouped_docs['group_id'].nunique()
print(f"  Number of groups: {n_groups}")
print(f"  Documents in groups: {len(grouped_docs):,}")
print(f"  Group sizes: min={grouped_docs['group_size'].min()}, max={grouped_docs['group_size'].max()}, mean={grouped_docs['group_size'].mean():.1f}")

# Distribution of group sizes (count unique groups per size)
print("\n  Group size distribution:")
size_dist = grouped_docs.drop_duplicates('group_id').groupby('group_size').size().sort_index(ascending=False)
for size, count in size_dist.items():
    print(f"    Size {size}: {count} groups ({count * size} docs)")


# Intermission — Save your results

Act 1 is finished. Before you explore, save your results. That gives you a copy to come back to, even if the session closes.

1. Click the **play button** on the code cell below.
2. Two files are saved, and your browser may start downloading them:
   - **[name]_grouped_v2.csv:** your original comments with the grouping columns added. Opens in Excel.
   - **[name]_grouped_v2.parquet:** the same data, for researchers working in Python or R.
3. **Done when:** you see "✓ Saved" twice, with the file names.

To find the files later, click the **folder icon** on the left side of Colab. Your files are listed there.

In [ ]:
original_name = Path(filepath).stem
output_csv = f"{original_name}_grouped_v2.csv"
# output_csv = f"{original_name}_grouped_numpy.csv"

# Select columns to export: original columns + new grouping columns
export_cols = list(df.columns) + [
    'group_id', 'group_size', 'group_label', 'is_identical',
    'J_to_rep', 'max_j_in_group', 'best_match_id',
    'is_group_rep', 'group_rep_id', 'family_id', 'family_size', 'family_anchor_row'
]
results[export_cols].to_csv(output_csv, index=False)

print(f"✓ Saved: {output_csv}")
print(f"  Rows: {len(results):,}")
print(f"  Original columns: {len(df.columns)}")
print(f"  New grouping columns: {len(export_cols) - len(df.columns)}")
print(f"  Total columns: {len(export_cols)}")

if IN_COLAB:
    files.download(output_csv)
else:
    print(f"  File saved to: {Path(output_csv).absolute()}")

output_parquet = f"{original_name}_grouper.parquet"
results[export_cols].to_parquet(output_parquet, index=False)
print(f"✓ Saved: {output_parquet}")
if IN_COLAB:
    files.download(output_parquet)

### Optional: timing log

The cell below adds one row to a timing log, recording how long this run took. You can skip it. It doesn't change your results.

In [ ]:
# Append this run's timings to a running log (one row per run)
RUN_LOG_CSV = 'grouping_run_timings_v4.csv'
log_path = Path(RUN_LOG_CSV)

try:
    import resource
    peak_rss_mb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024  # Linux (Colab) reports KB
except ImportError:
    peak_rss_mb = None  # not available on Windows

run_record = {
    'run_started': run_started_at.isoformat(timespec='seconds'),
    'source_file': Path(filepath).name,
    'environment': 'Google Colab' if IN_COLAB else 'Local Jupyter',
    'n_rows': n_docs,
    'n_unique_texts': int(is_first_seen_normalized.sum()),
    'FAMILY_J': FAMILY_J,
    # 'lsh_backend': LSH_BACKEND, # remove (no longer exists)
    'datasketch_version': datasketch.__version__,
    'peak_rss_mb': round(peak_rss_mb, 1) if peak_rss_mb is not None else None,
    'load_seconds': round(load_seconds, 3),
    **{f'{name}_seconds': secs for name, secs in run_stages.items()},
    'grouping_total_seconds': round(total_seconds, 3),
    'run_total_seconds': round(load_seconds + total_seconds, 3),
}
pd.DataFrame([run_record]).to_csv(log_path, mode='a', header=not log_path.exists(), index=False)

run_log = pd.read_csv(log_path)
display(run_log.tail(10))
print(f"✓ Timing logged to {log_path.absolute()} ({len(run_log)} runs total)")

if IN_COLAB:
    files.download(RUN_LOG_CSV)

# Act 2 — Explore your data

Now look at what Grouper found. The charts below are made from your results. Each chart is also saved as an image file, which you can find with the folder icon.

## Chart 1 — Grouped vs. ungrouped

The left bar compares ungrouped comments with grouped ones. The right chart shows the largest groups, split by similarity level. **Done when:** the chart appears.

In [ ]:
# Plot parameters
MIN_GROUP_SIZE_PLOT = 3
TOP_N_GROUPS = 40

# Prepare data for plotting
ungrouped = results[results['group_size'] == 1]
grouped = results[results['group_size'] > 1]

# Filter by size
large_groups = grouped[grouped['group_size'] >= MIN_GROUP_SIZE_PLOT]
large_group_ids = sorted(large_groups['group_id'].unique(),
                          key=lambda g: large_groups[large_groups['group_id'] == g].iloc[0]['group_size'],
                          reverse=True)[:TOP_N_GROUPS]

# Build plot data
plot_data = []
plot_labels = []

# Ungrouped bar
plot_labels.append('Ungrouped')
plot_data.append([len(ungrouped), 0, 0])

# Large groups
for gid in large_group_ids:
    g_data = grouped[grouped['group_id'] == gid]
    counts = g_data['group_label'].value_counts().to_dict()
    plot_labels.append(f"Grp {gid}")
    plot_data.append([
        counts.get('Grouped - Identical', 0),
        counts.get('Grouped - Highly similar', 0),
        counts.get('Grouped - Moderately similar', 0)
    ])

# Convert to arrays for stacking
plot_data = np.array(plot_data).T

# Create figure with 2 subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Colors - three shades of blue for the right plot
blue_colors = [plt.cm.Blues(0.85), plt.cm.Blues(0.65), plt.cm.Blues(0.4)]  # Dark, medium, light blue
bar_width = 0.6

# --- SUBPLOT 1: Ungrouped vs Grouped Summary ---
summary_labels = ['Ungrouped', 'Grouped']
summary_counts = [len(ungrouped), len(grouped)]
summary_colors = ['#999999', '#2E75B6']

ax1.bar(summary_labels, summary_counts, color=summary_colors, width=bar_width)
ax1.set_ylabel('Number of documents', fontsize=11, fontweight='bold')
ax1.set_title('Ungrouped vs Grouped', fontsize=12, fontweight='bold')
ax1.grid(axis='y', alpha=0.3, linestyle='--')
for i, v in enumerate(summary_counts):
    ax1.text(i, v + 10, str(v), ha='center', fontweight='bold')

# --- SUBPLOT 2: Group Composition (stacked bars) ---
group_plot_labels = plot_labels[1:]
group_plot_data = [row[1:] for row in plot_data]
x_pos = np.arange(len(group_plot_labels))

bottom = np.zeros(len(group_plot_labels))
labels_stack = ['Identical', 'Highly similar', 'Moderately similar']

for i, (row, label, color) in enumerate(zip(group_plot_data, labels_stack, blue_colors)):
    ax2.bar(x_pos, row, bar_width, label=label, bottom=bottom, color=color)
    bottom += np.array(row)

ax2.set_ylabel('Number of documents', fontsize=11, fontweight='bold')
#ax2.set_xlabel('Group (sorted by size)', fontsize=11, fontweight='bold')
ax2.set_title('Group Composition by Similarity Level', fontsize=12, fontweight='bold')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(group_plot_labels, rotation=45, ha='right', fontsize=9)
ax2.legend(loc='upper right', fontsize=10)
ax2.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()

# Save
plot_filename = f"{Path(filepath).stem}_grouped_plot.png"
plt.savefig(plot_filename, dpi=100, bbox_inches='tight')
print(f"✓ Saved plot: {Path(plot_filename).absolute()}")
if IN_COLAB:
    files.download(plot_filename)

plt.show()


## Chart 2 — The same chart on a log scale

The vertical axis is logarithmic, so each step up means ten times more comments. Small groups are easier to see next to very large ones.

In [ ]:
# Plot parameters
MIN_GROUP_SIZE_PLOT = 3
TOP_N_GROUPS = 40

# Prepare data for plotting
ungrouped = results[results['group_size'] == 1]
grouped = results[results['group_size'] > 1]

# Filter by size
large_groups = grouped[grouped['group_size'] >= MIN_GROUP_SIZE_PLOT]
large_group_ids = sorted(large_groups['group_id'].unique(),
                          key=lambda g: large_groups[large_groups['group_id'] == g].iloc[0]['group_size'],
                          reverse=True)[:TOP_N_GROUPS]

# Build plot data
plot_data = []
plot_labels = []

# Ungrouped bar
plot_labels.append('Ungrouped')
plot_data.append([len(ungrouped), 0, 0])

# Large groups
for gid in large_group_ids:
    g_data = grouped[grouped['group_id'] == gid]
    counts = g_data['group_label'].value_counts().to_dict()
    plot_labels.append(f"Grp {gid}")
    plot_data.append([
        counts.get('Grouped - Identical', 0),
        counts.get('Grouped - Highly similar', 0),
        counts.get('Grouped - Moderately similar', 0)
    ])

# Convert to arrays for stacking
plot_data = np.array(plot_data).T

# Create figure with 2 subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Colors - three shades of blue for the right plot
blue_colors = [plt.cm.Blues(0.85), plt.cm.Blues(0.65), plt.cm.Blues(0.4)]  # Dark, medium, light blue
bar_width = 0.6

# --- SUBPLOT 1: Ungrouped vs Grouped Summary ---
summary_labels = ['Ungrouped', 'Grouped']
summary_counts = [len(ungrouped), len(grouped)]
summary_colors = ['#999999', '#2E75B6']

ax1.bar(summary_labels, summary_counts, color=summary_colors, width=bar_width)
ax1.set_ylabel('Number of documents', fontsize=11, fontweight='bold')
ax1.set_title('Ungrouped vs Grouped', fontsize=12, fontweight='bold')
ax1.grid(axis='y', alpha=0.3, linestyle='--')
for i, v in enumerate(summary_counts):
    ax1.text(i, v + 10, str(v), ha='center', fontweight='bold')

# --- SUBPLOT 2: Group Composition (stacked bars) ---
group_plot_labels = plot_labels[1:]
group_plot_data = [row[1:] for row in plot_data]
x_pos = np.arange(len(group_plot_labels))

bottom = np.zeros(len(group_plot_labels))
labels_stack = ['Identical', 'Highly similar', 'Moderately similar']

for i, (row, label, color) in enumerate(zip(group_plot_data, labels_stack, blue_colors)):
    ax2.bar(x_pos, row, bar_width, label=label, bottom=bottom, color=color)
    bottom += np.array(row)

ax2.set_ylabel('Number of documents', fontsize=11, fontweight='bold')
#ax2.set_xlabel('Group (sorted by size)', fontsize=11, fontweight='bold')
ax2.set_title('Group Composition by Similarity Level', fontsize=12, fontweight='bold')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(group_plot_labels, rotation=45, ha='right', fontsize=9)
ax2.legend(loc='upper right', fontsize=10)
ax2.grid(axis='y', alpha=0.3, linestyle='--')
ax2.set_yscale('log')
ax2.set_yticks([1, 10, 100, 1000, 10000])
ax2.set_yticklabels(['1', '10', '100', '1000', '10000'])

plt.tight_layout()

# Save
plot_filename = f"{Path(filepath).stem}_grouped_ylog_plot.png"
plt.savefig(plot_filename, dpi=100, bbox_inches='tight')
print(f"✓ Saved plot: {Path(plot_filename).absolute()}")
if IN_COLAB:
    files.download(plot_filename)

plt.show()


## Chart 3 — Families

Each bar is a family. The colors show its groups, and darker colors are larger groups. Use it to see whether a family is mostly one big group or several smaller ones.

In [ ]:
TOP_N_FAMILIES = 25

fam_rows = results[results['family_id'].notna()]
fam_summary = fam_rows.groupby('family_id').agg(docs=('group_id', 'size'), groups=('group_id', 'nunique'))
fam_summary = fam_summary[fam_summary['groups'] > 1].sort_values('docs', ascending=False).head(TOP_N_FAMILIES)

if len(fam_summary) == 0:
    print("⊘ No families spanning 2+ groups to plot")
else:
    fig, ax = plt.subplots(figsize=(12, 0.45 * len(fam_summary) + 1.5))
    ypos = np.arange(len(fam_summary))[::-1]
    max_docs = fam_summary['docs'].max()

    for y, fid in zip(ypos, fam_summary.index):
        sizes = fam_rows[fam_rows['family_id'] == fid]['group_id'].value_counts()
        shades = plt.cm.Blues(np.linspace(0.85, 0.35, len(sizes)))
        left = 0
        for shade, n in zip(shades, sizes.values):
            ax.barh(y, n, left=left, color=shade, edgecolor='white', linewidth=0.6)
            left += n
        ax.text(left + max_docs * 0.01, y, f"{left} docs · {len(sizes)} groups", va='center', fontsize=9)

    ax.set_yticks(ypos)
    ax.set_yticklabels([f"Family {fid}" for fid in fam_summary.index], fontsize=9)
    ax.set_xlabel('Documents', fontsize=11, fontweight='bold')
    ax.set_title('Families: member groups stacked by size (darkest = largest group)', fontsize=12, fontweight='bold')
    ax.set_xlim(0, max_docs * 1.25)
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    plt.tight_layout()

    family_plot_filename = f"{Path(filepath).stem}_family_plot.png"
    plt.savefig(family_plot_filename, dpi=100, bbox_inches='tight')
    print(f"✓ Saved family plot: {Path(family_plot_filename).absolute()}")
    if IN_COLAB:
        files.download(family_plot_filename)
    plt.show()


## Chart 4 — Family breakdown

Shows the groups in each family side by side, stacked by similarity level. It also saves a table (ending in _family_composition.csv) that you can open in Excel.

In [ ]:
# Family composition: each family's groups side by side, stacked by similarity level
LEVELS = ['Grouped - Identical', 'Grouped - Highly similar', 'Grouped - Moderately similar']
LEVEL_NAMES = ['Identical', 'Highly similar', 'Moderately similar']
LEVEL_COLORS = [plt.cm.Blues(0.85), plt.cm.Blues(0.65), plt.cm.Blues(0.4)]
FAMILY_GAP = 1.0  # space between families on the x-axis

fam_rows = results[results['family_id'].notna()]

if len(fam_rows) == 0:
    print("⊘ No families to plot")
else:
    fam_order = fam_rows.groupby('family_id').size().sort_values(ascending=False).index.tolist()

    table_rows, bars, fam_spans = [], [], []
    x = 0.0
    for fid in fam_order:
        fam = fam_rows[fam_rows['family_id'] == fid]
        anchor_group = results.at[int(fam['family_anchor_row'].iloc[0]), 'group_id']
        group_order = fam.groupby('group_id')['group_size'].first().sort_values(ascending=False).index
        start = x
        for gid in group_order:
            g = fam[fam['group_id'] == gid]
            counts = g['group_label'].value_counts()
            level_counts = [int(counts.get(level, 0)) for level in LEVELS]
            total = int(len(g))
            bars.append((x, gid, level_counts, total))
            table_rows.append({
                'family_id': int(fid),
                'group_id': int(gid),
                'family_anchor_group': bool(gid == anchor_group),
                'comments': total,
                'identical': level_counts[0],
                'highly_similar': level_counts[1],
                'moderately_similar': level_counts[2],
            })
            x += 1
        fam_spans.append((fid, start, x - 1))
        x += FAMILY_GAP

    fig_w = max(10, 0.45 * len(bars) + 3)
    fig, ax = plt.subplots(figsize=(fig_w, 6.5))
    xs = np.array([b[0] for b in bars])
    bottom = np.zeros(len(bars))
    for k, (name, color) in enumerate(zip(LEVEL_NAMES, LEVEL_COLORS)):
        vals = np.array([b[2][k] for b in bars], dtype=float)
        mask = vals > 0
        ax.bar(xs[mask], vals[mask], 0.7, bottom=bottom[mask], color=color, label=name)
        bottom += vals

    for xp, gid, _, total in bars:
        ax.text(xp, total * 1.15, f"{total:,}", ha='center', va='bottom', fontsize=8)

    for fid, s, e in fam_spans:
        ax.text((s + e) / 2, 1.02, f"Family {fid}", transform=ax.get_xaxis_transform(),
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    for _, _, e in fam_spans[:-1]:
        ax.axvline(e + FAMILY_GAP / 2, color='#cccccc', linestyle='--', linewidth=0.8)

    ax.set_yscale('log')
    ax.set_ylim(1, bottom.max() * 3)
    ax.set_ylabel('Number of comments (log scale)', fontsize=11, fontweight='bold')
    ax.set_xticks(xs)
    ax.set_xticklabels([f"Grp {b[1]}" for b in bars], rotation=45, ha='right', fontsize=9)
    ax.set_title('Family composition: groups in each family, stacked by similarity level',
                 fontsize=12, fontweight='bold', pad=28)
    ax.legend(loc='upper right', fontsize=10)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    plt.tight_layout()

    family_table = pd.DataFrame(table_rows)
    display(family_table)

    table_filename = f"{Path(filepath).stem}_family_composition.csv"
    family_table.to_csv(table_filename, index=False)
    print(f"✓ Saved family table: {Path(table_filename).absolute()}")

    plot_filename = f"{Path(filepath).stem}_family_composition_plot.png"
    plt.savefig(plot_filename, dpi=100, bbox_inches='tight')
    print(f"✓ Saved family composition plot: {Path(plot_filename).absolute()}")
    if IN_COLAB:
        files.download(table_filename)
        files.download(plot_filename)
    plt.show()

# Act 2 (continued) — Browse the comments

Now read the actual comments. You'll load your saved results, then browse families or groups and compare comments side by side. Start with Step A, then use Step B (families) or Step C (groups).

## Step A — Load your saved results

Use this step if you're starting in a new session, or if you want to use the results you saved in the Intermission.

1. Click the **play button** on the code cell below.
2. Click **Choose Files** (or **Browse**), and select your **_grouped_v2.csv** file.
3. **Done when:** you see "Loaded" with the number of rows, and the line after it shows **0**. That 0 is a check that every grouped comment has a valid label.

In [ ]:
label_order = ['Ungrouped', 'Grouped - Identical', 'Grouped - Highly similar', 'Grouped - Moderately similar']

if IN_COLAB:
    print("Click 'Choose Files' to upload your *_grouped.csv file:")
    uploaded = files.upload()
    grouped_path = list(uploaded.keys())[0]
else:
    grouped_path = input("Enter path to your grouped CSV: ")

results = pd.read_csv(grouped_path, dtype=str, keep_default_na=False)
for col in ['group_id', 'group_size', 'best_match_id', 'group_rep_id']:
    results[col] = results[col].astype(int)
for col in ['family_id', 'family_size', 'family_anchor_row']:
    results[col] = pd.to_numeric(results[col].replace('', np.nan)).astype('Int64')
for col in ['J_to_rep', 'max_j_in_group']:
    results[col] = results[col].astype(float)
for col in ['is_identical', 'is_group_rep']:
    results[col] = results[col].map({'True': True, 'False': False})
results['group_label'] = pd.Categorical(results['group_label'], categories=label_order, ordered=True)

comment_col = detect_comment_column(results)
id_col = detect_id_column(results)
normalized_texts = np.array(
    [normalize_text(t) for t in tqdm(results[comment_col], desc='Normalizing', unit='doc')],
    dtype=object
)

print(f"Loaded {len(results):,} rows")

In [ ]:
bad = results[(results['group_size'] > 1) & (results['group_label'] == 'Ungrouped')]
print(len(bad))
print(bad[['group_id', 'group_size', 'J_to_rep', 'max_j_in_group', 'is_identical']])


## Step B — Browse families

A family is a set of groups that are variations of the same template. Use the **Family** dropdown to pick a family. Then use **Compare with** to pick a group from that family. The two comments are shown side by side, with their differences highlighted.

The cell that shows the tool is the one after the next heading.

In [ ]:
build_family_explorer(results, comment_col, id_col, results['family_anchor_row'], normalized_texts)

## Step C — Browse groups

Use the **Group** dropdown to pick any group. Then pick two comments with **Doc 1** and **Doc 2**. They are shown side by side, with the differing words highlighted and a similarity score. Use this to check that the grouping makes sense before you rely on it.

Run the cell below to show the group explorer.

In [ ]:
build_explorer(results, comment_col, id_col)